# LunarSite Stage 2: v1 vs v2 Test Evaluation (Self-Contained)

**Goal:** Fair apples-to-apples comparison of v1 (ResNet-34 + Dice+CE) vs v2 (ResNet-50 + Focal+Dice + class weights).

## How to run
1. Create new Kaggle notebook, File -> Import Notebook, upload this .ipynb
2. Right sidebar: Accelerator = **GPU T4 x2**, Internet = **On**
3. Click **Run All** (play button above cells, NOT Save Version)
4. Wait ~25 minutes
5. Download `test_comparison.json` from `/kaggle/working/` in the Output tab

**No manual dataset attachment required** - kagglehub auto-downloads both the lunar landscape dataset and your checkpoint dataset (`encinas88/lunarsite-checkpoints`).

## What it measures
- 2 models (v1 ResNet-34, v2 ResNet-50) x 3 eval types (standard, flip TTA, multi-scale TTA) = **6 evaluation runs**
- Per run: mean IoU, per-class IoU, mean Dice, per-image distribution, confusion matrix, inference time


In [ ]:
# === INSTALL DEPENDENCIES === #
!pip install -q segmentation-models-pytorch albumentations kagglehub

In [ ]:
import json
import time
from pathlib import Path

import albumentations as A
import cv2
import kagglehub
import numpy as np
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

SEED = 42
NC = 4
INPUT_SIZE = 480
CLASS_NAMES = ['background', 'small_rocks', 'large_rocks', 'sky']
COLOR_TO_CLASS = {(0, 0, 0): 0, (255, 0, 0): 1, (0, 255, 0): 2, (0, 0, 255): 3}

V1_REFERENCE = {
    'test_mean_iou': 0.8425,
    'test_per_class_iou': {
        'background': 0.9754,
        'small_rocks': 0.9748,
        'large_rocks': 0.7092,
        'sky': 0.7104,
    },
}

# === DOWNLOAD CHECKPOINTS via kagglehub === #
# Auto-downloads encinas88/lunarsite-checkpoints (your private dataset with both .pt files)
print('
Downloading checkpoint dataset...')
ckpt_root = Path(kagglehub.dataset_download('encinas88/lunarsite-checkpoints'))
print(f'Checkpoint dataset: {ckpt_root}')

# Find the two checkpoint files inside
V1_CKPT = None
V2_CKPT = None
for pt in ckpt_root.glob('**/*.pt'):
    name = pt.name.lower()
    if 'segmenter' in name and V1_CKPT is None:
        V1_CKPT = str(pt)
        print(f'  v1 (ResNet-34): {V1_CKPT}')
    elif 'resnet50' in name and V2_CKPT is None:
        V2_CKPT = str(pt)
        print(f'  v2 (ResNet-50): {V2_CKPT}')

assert V1_CKPT is not None, f'Could not find v1 checkpoint in {ckpt_root}. Files: {list(ckpt_root.glob("**/*.pt"))}'
assert V2_CKPT is not None, f'Could not find v2 checkpoint in {ckpt_root}. Files: {list(ckpt_root.glob("**/*.pt"))}'

OUT_DIR = '/kaggle/working'


## Dataset + Test Split

Replicates the exact deterministic split from training notebooks (SEED=42, 80/10/10).

In [ ]:
# Download lunar landscape dataset via kagglehub
print('Downloading lunar landscape dataset...')
dataset_path = Path(kagglehub.dataset_download('romainpessia/artificial-lunar-rocky-landscape-dataset'))
image_dir = dataset_path / 'images' / 'render'
mask_dir = dataset_path / 'images' / 'clean'
print(f'Dataset: {dataset_path}')
assert image_dir.exists(), f'image_dir missing: {image_dir}'
assert mask_dir.exists(), f'mask_dir missing: {mask_dir}'

# Deterministic split (EXACT same as training: SEED=42, 80/10/10)
all_imgs = sorted(image_dir.glob('render*.png'))
all_masks = sorted(mask_dir.glob('clean*.png'))
n = len(all_imgs)
perm = np.random.RandomState(SEED).permutation(n).tolist()
nt, nv = int(n * 0.8), int(n * 0.1)
train_idx, val_idx, test_idx = perm[:nt], perm[nt:nt+nv], perm[nt+nv:]
print(f'Total: {n} | Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}')

test_img_paths = [all_imgs[i] for i in test_idx]
test_mask_paths = [all_masks[i] for i in test_idx]


In [ ]:
class LunarTestDataset(Dataset):
    def __init__(self, image_paths, mask_paths, size=INPUT_SIZE):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = A.Compose([A.CenterCrop(size, size)])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.cvtColor(cv2.imread(str(self.image_paths[idx])), cv2.COLOR_BGR2RGB)
        msk = cv2.cvtColor(cv2.imread(str(self.mask_paths[idx])), cv2.COLOR_BGR2RGB)
        h, w = msk.shape[:2]
        mask = np.zeros((h, w), dtype=np.int64)
        for color, cls in COLOR_TO_CLASS.items():
            mask[np.all(msk == color, axis=-1)] = cls
        t = self.transform(image=img, mask=mask)
        img, mask = t['image'], t['mask']
        return {
            'image': torch.from_numpy(img.transpose(2, 0, 1).astype(np.float32) / 255.0),
            'mask': torch.from_numpy(mask.astype(np.int64)),
        }

test_set = LunarTestDataset(test_img_paths, test_mask_paths, size=INPUT_SIZE)
test_loader = DataLoader(test_set, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
print(f'Test set ready: {len(test_set)} images at {INPUT_SIZE}x{INPUT_SIZE}')

## Model Builders + Checkpoint Loader

In [ ]:
def build_unet(encoder_name):
    return smp.Unet(
        encoder_name=encoder_name,
        encoder_weights='imagenet',
        in_channels=3,
        classes=NC,
    )

def load_checkpoint(model, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        state = ckpt['model_state_dict']
        print(f'  Loaded epoch {ckpt.get("epoch", "?")}, best metric {ckpt.get("best_metric", "?")}')
    elif isinstance(ckpt, dict) and 'state_dict' in ckpt:
        state = ckpt['state_dict']
        print(f'  Loaded from state_dict key')
    elif isinstance(ckpt, dict) and all(isinstance(v, torch.Tensor) for v in ckpt.values()):
        state = ckpt
        print(f'  Loaded raw state_dict ({len(state)} tensors)')
    else:
        state = ckpt
        print(f'  Loaded (unknown format, trying direct)')
    model.load_state_dict(state)
    model.to(device).eval()
    return model

print('Loading v1 (ResNet-34)...')
model_v1 = build_unet('resnet34')
model_v1 = load_checkpoint(model_v1, V1_CKPT)
print(f'  Params: {sum(p.numel() for p in model_v1.parameters()):,}')

print()
print('Loading v2 (ResNet-50)...')
model_v2 = build_unet('resnet50')
model_v2 = load_checkpoint(model_v2, V2_CKPT)
print(f'  Params: {sum(p.numel() for p in model_v2.parameters()):,}')

## Inference Helpers: Standard, Flip TTA, Multi-Scale TTA

In [ ]:
@torch.no_grad()
def predict_standard(model, x):
    return F.softmax(model(x), dim=1)

@torch.no_grad()
def predict_tta_flip(model, x):
    probs = F.softmax(model(x), dim=1)
    probs = probs + torch.flip(F.softmax(model(torch.flip(x, dims=[3])), dim=1), dims=[3])
    probs = probs + torch.flip(F.softmax(model(torch.flip(x, dims=[2])), dim=1), dims=[2])
    probs = probs + torch.flip(F.softmax(model(torch.flip(x, dims=[2, 3])), dim=1), dims=[2, 3])
    return probs / 4.0

@torch.no_grad()
def predict_tta_multiscale(model, x, scales=(0.75, 1.0, 1.25)):
    _, _, H, W = x.shape
    accum = torch.zeros(x.size(0), NC, H, W, device=x.device)
    for s in scales:
        if s == 1.0:
            xs = x
        else:
            nh, nw = int(H * s), int(W * s)
            nh = ((nh + 31) // 32) * 32
            nw = ((nw + 31) // 32) * 32
            xs = F.interpolate(x, size=(nh, nw), mode='bilinear', align_corners=False)
        p = F.softmax(model(xs), dim=1)
        p = p + torch.flip(F.softmax(model(torch.flip(xs, dims=[3])), dim=1), dims=[3])
        p = p + torch.flip(F.softmax(model(torch.flip(xs, dims=[2])), dim=1), dims=[2])
        p = p + torch.flip(F.softmax(model(torch.flip(xs, dims=[2, 3])), dim=1), dims=[2, 3])
        p = p / 4.0
        if p.shape[-2:] != (H, W):
            p = F.interpolate(p, size=(H, W), mode='bilinear', align_corners=False)
        accum += p
    return accum / len(scales)

PRED_FNS = {
    'standard': predict_standard,
    'tta_flip': predict_tta_flip,
    'tta_multiscale': predict_tta_multiscale,
}
print('TTA functions ready.')

## Metric + Evaluation Loop

In [ ]:
def compute_iou_from_confusion(conf):
    nc = conf.shape[0]
    ious = []
    dices = []
    for c in range(nc):
        tp = conf[c, c].item()
        fp = conf[:, c].sum().item() - tp
        fn = conf[c, :].sum().item() - tp
        union = tp + fp + fn
        ious.append(tp / union if union > 0 else 0.0)
        denom = 2 * tp + fp + fn
        dices.append(2 * tp / denom if denom > 0 else 0.0)
    return ious, dices

def evaluate(model, loader, predict_fn, nc=NC, tag=''):
    model.eval()
    conf = torch.zeros(nc, nc, dtype=torch.long)
    per_image_miou = []

    if device.type == 'cuda':
        torch.cuda.synchronize()
    t_start = time.time()
    n_images = 0

    for batch in loader:
        imgs = batch['image'].to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)
        probs = predict_fn(model, imgs)
        preds = probs.argmax(1)

        preds_cpu = preds.cpu()
        masks_cpu = masks.cpu()

        for i in range(preds_cpu.size(0)):
            p = preds_cpu[i]
            m = masks_cpu[i]
            img_ious = []
            for c in range(nc):
                pc, mc = (p == c), (m == c)
                inter = (pc & mc).sum().item()
                union = (pc | mc).sum().item()
                if union > 0:
                    img_ious.append(inter / union)
            if img_ious:
                per_image_miou.append(float(np.mean(img_ious)))

        flat_true = masks_cpu.view(-1)
        flat_pred = preds_cpu.view(-1)
        idx = flat_true * nc + flat_pred
        bc = torch.bincount(idx, minlength=nc * nc)
        conf += bc.reshape(nc, nc)

        n_images += imgs.size(0)

    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.time() - t_start

    ious, dices = compute_iou_from_confusion(conf)
    per_class_iou = {CLASS_NAMES[c]: ious[c] for c in range(nc)}
    per_class_dice = {CLASS_NAMES[c]: dices[c] for c in range(nc)}

    arr = np.array(per_image_miou)
    result = {
        'mean_iou': float(np.mean(ious)),
        'mean_dice': float(np.mean(dices)),
        'per_class_iou': per_class_iou,
        'per_class_dice': per_class_dice,
        'per_image_miou': {
            'mean': float(arr.mean()),
            'std': float(arr.std()),
            'min': float(arr.min()),
            'max': float(arr.max()),
            'p10': float(np.percentile(arr, 10)),
            'p50': float(np.percentile(arr, 50)),
            'p90': float(np.percentile(arr, 90)),
        },
        'confusion_matrix': conf.tolist(),
        'total_time_s': elapsed,
        'time_per_image_ms': (elapsed / n_images) * 1000,
        'n_images': n_images,
    }
    miou = result['mean_iou']
    dice = result['mean_dice']
    pi_mean = result['per_image_miou']['mean']
    pi_std = result['per_image_miou']['std']
    pi_min = result['per_image_miou']['min']
    tpi = result['time_per_image_ms']
    print(f'  {tag}: mIoU={miou:.4f} | dice={dice:.4f} | per-img mean={pi_mean:.4f} (std={pi_std:.4f}, min={pi_min:.4f}) | {tpi:.1f} ms/img')
    return result

print('Evaluation loop ready.')

## Run Full Evaluation Matrix

In [ ]:
results = {}

for model_tag, model in [('v1_resnet34', model_v1), ('v2_resnet50', model_v2)]:
    results[model_tag] = {}
    print()
    print('=' * 70)
    print(f'MODEL: {model_tag}')
    print('=' * 70)
    for eval_tag, pred_fn in PRED_FNS.items():
        print(f'\n  Running {eval_tag}...')
        results[model_tag][eval_tag] = evaluate(model, test_loader, pred_fn, tag=eval_tag)

results['_v1_reference_test_results_json'] = V1_REFERENCE

out_path = f'{OUT_DIR}/test_comparison.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\n\nSaved: {out_path}')

## Summary Table + Final Verdict

In [ ]:
print()
print('=' * 85)
print('  LUNARSITE STAGE 2: v1 vs v2 TEST EVALUATION')
print('=' * 85)

header = f'{"":<24} | {"Mean IoU":>10} | {"Mean Dice":>10} | {"Per-img mean":>13} | {"Worst img":>10} | {"ms/img":>8}'
print(header)
print('-' * len(header))

for model_tag in ['v1_resnet34', 'v2_resnet50']:
    for eval_tag in ['standard', 'tta_flip', 'tta_multiscale']:
        r = results[model_tag][eval_tag]
        label = f'{model_tag} / {eval_tag}'
        mi = r['mean_iou']
        md = r['mean_dice']
        pim = r['per_image_miou']['mean']
        pimin = r['per_image_miou']['min']
        tpi = r['time_per_image_ms']
        print(f'{label:<24} | {mi:>10.4f} | {md:>10.4f} | {pim:>13.4f} | {pimin:>10.4f} | {tpi:>8.1f}')
    print('-' * len(header))

print(f'\nv1 reference (from test_results.json, pre-TTA): mIoU={V1_REFERENCE["test_mean_iou"]:.4f}')

print()
print('=' * 85)
print('  PER-CLASS IoU (best eval for each model)')
print('=' * 85)

def best_eval(model_tag):
    return max(['standard', 'tta_flip', 'tta_multiscale'], key=lambda e: results[model_tag][e]['mean_iou'])

for model_tag in ['v1_resnet34', 'v2_resnet50']:
    best = best_eval(model_tag)
    r = results[model_tag][best]
    print(f'\n{model_tag} (best: {best}, mIoU={r["mean_iou"]:.4f}):')
    for cname in CLASS_NAMES:
        v1_ref = V1_REFERENCE['test_per_class_iou'].get(cname, None)
        delta = ''
        if v1_ref is not None and model_tag == 'v2_resnet50':
            d = r['per_class_iou'][cname] - v1_ref
            delta = f'  (vs v1 ref: {v1_ref:.4f}, delta {d:+.4f})'
        print(f'  {cname:<15}: {r["per_class_iou"][cname]:.4f}{delta}')

print()
print('=' * 85)
print('  CONFUSION MATRIX - normalized by true class (best eval)')
print('=' * 85)

for model_tag in ['v1_resnet34', 'v2_resnet50']:
    best = best_eval(model_tag)
    conf = np.array(results[model_tag][best]['confusion_matrix'])
    row_totals = conf.sum(axis=1, keepdims=True)
    conf_norm = conf / np.maximum(row_totals, 1)
    print(f'\n{model_tag} ({best}):')
    header_line = '  true \\ pred     | ' + ' | '.join(f'{c:>12}' for c in CLASS_NAMES)
    print(header_line)
    for i, cname in enumerate(CLASS_NAMES):
        row_str = ' | '.join(f'{conf_norm[i, j]:>12.4f}' for j in range(NC))
        print(f'  {cname:<15} | {row_str}')

In [ ]:
v1_best = max(results['v1_resnet34'][e]['mean_iou'] for e in ['standard', 'tta_flip', 'tta_multiscale'])
v2_best = max(results['v2_resnet50'][e]['mean_iou'] for e in ['standard', 'tta_flip', 'tta_multiscale'])
v1_std = results['v1_resnet34']['standard']['mean_iou']
v2_std = results['v2_resnet50']['standard']['mean_iou']

print()
print('=' * 85)
print('  FINAL VERDICT')
print('=' * 85)
print(f'v1 standard (no TTA):   {v1_std:.4f}')
print(f'v1 best (any eval):     {v1_best:.4f}')
print(f'v2 standard (no TTA):   {v2_std:.4f}')
print(f'v2 best (any eval):     {v2_best:.4f}')
print()
print(f'Apples-to-apples (std vs std):            v2 - v1 = {v2_std - v1_std:+.4f}')
print(f'Best vs best (TTA allowed both sides):    v2 - v1 = {v2_best - v1_best:+.4f}')
print()

if abs(v2_best - v1_best) < 0.003:
    winner = 'TIE (within noise)'
    margin = abs(v2_best - v1_best)
elif v2_best > v1_best:
    winner = 'v2 (ResNet-50 + Focal+Dice + class weights)'
    margin = v2_best - v1_best
else:
    winner = 'v1 (ResNet-34 + Dice+CE)'
    margin = v1_best - v2_best

print(f'WINNER: {winner}')
print(f'Margin: {margin:.4f} mIoU')

v1_tta_gain = results['v1_resnet34']['tta_flip']['mean_iou'] - v1_std
v2_tta_gain = results['v2_resnet50']['tta_flip']['mean_iou'] - v2_std
v1_ms_gain = results['v1_resnet34']['tta_multiscale']['mean_iou'] - results['v1_resnet34']['tta_flip']['mean_iou']
v2_ms_gain = results['v2_resnet50']['tta_multiscale']['mean_iou'] - results['v2_resnet50']['tta_flip']['mean_iou']

print()
print(f'Flip TTA value:          v1 +{v1_tta_gain:.4f} | v2 +{v2_tta_gain:.4f}')
print(f'Multi-scale extra value: v1 +{v1_ms_gain:.4f} | v2 +{v2_ms_gain:.4f}')
print()
print('RECOMMENDATION:')
max_ms = max(v1_ms_gain, v2_ms_gain)
if max_ms < 0.005:
    print('  Multi-scale TTA gain is negligible (<0.5 pts). Use flip-only TTA in production.')
elif max_ms < 0.015:
    print('  Multi-scale TTA gives modest improvement. Use for benchmark numbers,')
    print('  flip-only for Streamlit demo (~3x faster).')
else:
    print('  Multi-scale TTA gives real improvement. Worth the compute for production.')

print()
print('Done. Results saved to /kaggle/working/test_comparison.json')